# 📦 Damaged Box Detection - Complete Training Pipeline

**Dataset:** https://universe.roboflow.com/project-33xgh/damaged-box-detection

**What this notebook does:**
1. ✅ Downloads the COMPLETE damaged box dataset from Roboflow
2. ✅ Includes ALL images (torn, crushed, damaged - everything!)
3. ✅ Trains YOLOv8 model for real-time detection
4. ✅ Exports to ONNX for Jetson Orin deployment

**Requirements:**
- Free Roboflow account (get API key at https://app.roboflow.com/)
- Google Colab with GPU (Runtime → Change runtime type → GPU)

---

## 🔑 Step 1: Get Your Roboflow API Key

**Quick Setup (2 minutes):**

1. Go to https://app.roboflow.com/
2. Click **"Sign Up"** (free, no credit card)
3. After login, go to **Settings** (⚙️ icon)
4. Click **"Roboflow API"**
5. Copy your **Private API Key**
6. Paste it in the cell below

**Your API key looks like:** `aBcDeFgHiJkLmNoPqRsTuVwXyZ123456`

In [ ]:
# Install required packages
!pip install -q roboflow ultralytics

print("✅ Packages installed")

In [ ]:
# 👇 PASTE YOUR API KEY HERE 👇
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

# Verify
if ROBOFLOW_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ STOP! Please replace YOUR_API_KEY_HERE with your actual Roboflow API key")
    print("\n📍 Get your FREE API key:")
    print("   1. Go to https://app.roboflow.com/")
    print("   2. Sign up (free)")
    print("   3. Settings → Roboflow API → Copy key")
    print("   4. Paste it above and run this cell again")
else:
    print("✅ API key configured!")
    print(f"   Key: {ROBOFLOW_API_KEY[:8]}...{ROBOFLOW_API_KEY[-4:]}")

## 📥 Step 2: Download the Damaged Box Dataset

**About the dataset:**
- Source: https://universe.roboflow.com/project-33xgh/damaged-box-detection
- Contains: Damaged, torn, crushed cardboard boxes
- Format: YOLO format with bounding box annotations
- Classes: Damaged boxes (various types)

**Note:** The URL has `?queryText=torn` which is just a BROWSER FILTER.
The API download below gets **ALL images** regardless of tags!

In [ ]:
from roboflow import Roboflow

print("🚀 Initializing Roboflow...\n")

# Initialize
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Access the damaged-box-detection project
print("📦 Accessing project: damaged-box-detection\n")
project = rf.workspace("project-33xgh").project("damaged-box-detection")

# Get version 1 (change to 2, 3, etc. if there are multiple versions)
version = project.version(1)

print(f"📊 Project Details:")
print(f"   Name: {project.name}")
print(f"   Version: {version.version}")
print(f"   Splits: {version.splits}")
print(f"\n💡 Downloading COMPLETE dataset (ALL images: torn, crushed, damaged)")

In [ ]:
# Download in YOLOv8 format
print("\n⏳ Downloading dataset (this may take 2-5 minutes)...\n")
print("="*70)

dataset = version.download("yolov8")

print("="*70)
print(f"\n✅ Dataset downloaded successfully!")
print(f"📁 Location: {dataset.location}")
print(f"📄 Config: {dataset.location}/data.yaml")

## 📊 Step 3: Verify Dataset & Explore

In [ ]:
# Verify complete download
from pathlib import Path

dataset_path = Path(dataset.location)

print("🔍 Verifying download...\n")
print("="*70)

# Count images
train_images = list((dataset_path / "train" / "images").glob("*"))
valid_images = list((dataset_path / "valid" / "images").glob("*"))
test_images = list((dataset_path / "test" / "images").glob("*")) if (dataset_path / "test").exists() else []

total = len(train_images) + len(valid_images) + len(test_images)

print(f"📊 Image Counts:")
print(f"   Training:   {len(train_images):4d} images")
print(f"   Validation: {len(valid_images):4d} images")
print(f"   Test:       {len(test_images):4d} images")
print(f"   ─────────────────────")
print(f"   TOTAL:      {total:4d} images")
print(f"\n✅ ALL images downloaded (includes torn, crushed, damaged, etc.)")
print("="*70)

In [ ]:
# View dataset configuration
import yaml

with open(dataset_path / "data.yaml", 'r') as f:
    config = yaml.safe_load(f)

print("📄 Dataset Configuration (data.yaml):\n")
print("="*70)
for key, value in config.items():
    print(f"   {key}: {value}")
print("="*70)

## 🖼️ Step 4: Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

def show_samples(num_samples=6):
    """Display random training images with bounding boxes"""
    
    image_dir = dataset_path / "train" / "images"
    label_dir = dataset_path / "train" / "labels"
    
    # Get random samples
    all_images = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png"))
    samples = random.sample(all_images, min(num_samples, len(all_images)))
    
    # Load class names
    with open(dataset_path / "data.yaml") as f:
        class_names = yaml.safe_load(f)['names']
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(samples):
        # Load image
        img = Image.open(img_path)
        ax = axes[idx]
        ax.imshow(img)
        
        # Load labels
        label_path = label_dir / (img_path.stem + ".txt")
        
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Convert YOLO to pixel coordinates
                    img_w, img_h = img.size
                    x1 = (x_center - width/2) * img_w
                    y1 = (y_center - height/2) * img_h
                    box_w = width * img_w
                    box_h = height * img_h
                    
                    # Draw box
                    rect = patches.Rectangle(
                        (x1, y1), box_w, box_h,
                        linewidth=3,
                        edgecolor='red',
                        facecolor='none'
                    )
                    ax.add_patch(rect)
                    
                    # Label
                    label = class_names[int(class_id)]
                    ax.text(
                        x1, y1-10,
                        label,
                        color='white',
                        fontsize=12,
                        weight='bold',
                        bbox=dict(facecolor='red', alpha=0.8, pad=2)
                    )
        
        ax.axis('off')
        ax.set_title(f"Sample {idx+1}", fontsize=14, weight='bold')
    
    plt.tight_layout()
    plt.show()

print("🖼️ Displaying sample images with annotations...\n")
show_samples(6)

## 🤖 Step 5: Train YOLOv8 Model

**Model Choice:**
- YOLOv8n (nano) - Fastest, best for Jetson Orin
- YOLOv8s (small) - Better accuracy, slightly slower
- YOLOv8m (medium) - Best accuracy, slower

**Training Settings:**
- 50 epochs (increase to 100 for better results)
- 640x640 image size
- Batch size 16 (adjust based on GPU memory)

In [ ]:
# Check GPU availability
import torch
from ultralytics import YOLO

print("🖥️ System Check:\n")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✅ GPU ready for training!")
else:
    print("\n⚠️ No GPU detected!")
    print("   Go to: Runtime → Change runtime type → Hardware accelerator → GPU")
    print("   Then restart this cell")

In [ ]:
# Load YOLOv8 nano model (fastest for Jetson)
model = YOLO('yolov8n.pt')

print("✅ YOLOv8n model loaded")
print("   (Change to 'yolov8s.pt' or 'yolov8m.pt' for higher accuracy)")

In [ ]:
# Train the model
print("🚀 Starting training...\n")
print("="*70)
print("⏱️ Estimated time: 30-60 minutes (50 epochs on GPU)")
print("📊 You can monitor progress in real-time below")
print("="*70)
print()

results = model.train(
    data=str(dataset_path / "data.yaml"),
    epochs=50,                    # Increase to 100 for better results
    imgsz=640,
    batch=16,                     # Reduce to 8 if GPU runs out of memory
    name='damaged_box_yolov8',
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=10,
    save=True,
    plots=True,
    verbose=True
)

print("\n" + "="*70)
print("✅ Training complete!")
print("="*70)

## 📊 Step 6: View Training Results

In [ ]:
# Display training metrics
from IPython.display import Image as IPImage, display
from pathlib import Path

results_dir = Path("runs/detect/damaged_box_yolov8")

print("📈 Training Results:\n")
print("="*70)

# Training curves
if (results_dir / "results.png").exists():
    print("\n1️⃣ Training Curves (Loss, mAP, Precision, Recall):\n")
    display(IPImage(filename=str(results_dir / "results.png")))

# Confusion matrix
if (results_dir / "confusion_matrix.png").exists():
    print("\n2️⃣ Confusion Matrix:\n")
    display(IPImage(filename=str(results_dir / "confusion_matrix.png")))

# F1 curve
if (results_dir / "F1_curve.png").exists():
    print("\n3️⃣ F1 Score Curve:\n")
    display(IPImage(filename=str(results_dir / "F1_curve.png")))

print("\n" + "="*70)

## 🧪 Step 7: Test the Model

In [ ]:
# Load best model and validate
best_model = YOLO(str(results_dir / "weights" / "best.pt"))

print("🔍 Running validation on test set...\n")
metrics = best_model.val()

print("\n" + "="*70)
print("📊 Model Performance Metrics:")
print("="*70)
print(f"   mAP50:      {metrics.box.map50:.3f}  (Higher is better, max 1.0)")
print(f"   mAP50-95:   {metrics.box.map:.3f}  (Higher is better, max 1.0)")
print(f"   Precision:  {metrics.box.mp:.3f}  (How many detections are correct)")
print(f"   Recall:     {metrics.box.mr:.3f}  (How many damages are found)")
print("="*70)

if metrics.box.map50 > 0.8:
    print("\n✅ Excellent model performance!")
elif metrics.box.map50 > 0.6:
    print("\n✅ Good model performance!")
else:
    print("\n⚠️ Model needs more training. Try increasing epochs to 100.")

In [ ]:
# Test on sample images
import matplotlib.pyplot as plt

test_imgs = list((dataset_path / "valid" / "images").glob("*"))[:6]

print("🖼️ Testing on validation images...\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(test_imgs):
    # Predict
    results = best_model.predict(str(img_path), conf=0.5, verbose=False)
    
    # Plot
    annotated = results[0].plot()
    axes[idx].imshow(annotated[..., ::-1])  # BGR to RGB
    axes[idx].axis('off')
    axes[idx].set_title(f"Detection {idx+1}", fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

print("✅ Predictions complete!")

## 📤 Step 8: Export for Jetson Orin

In [ ]:
# Export to ONNX
print("📦 Exporting model to ONNX format...\n")
print("="*70)

onnx_file = best_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    dynamic=False
)

print("="*70)
print(f"\n✅ ONNX model exported: {onnx_file}")
print(f"\n📋 File sizes:")

import os
pt_size = os.path.getsize(results_dir / "weights" / "best.pt") / (1024*1024)
onnx_size = os.path.getsize(onnx_file) / (1024*1024)

print(f"   PyTorch (.pt):  {pt_size:.2f} MB")
print(f"   ONNX (.onnx):   {onnx_size:.2f} MB")

## 🚀 Jetson Orin Deployment Instructions

**After downloading the ONNX file, on your Jetson Orin:**

```bash
# Convert ONNX to TensorRT (FP16 for 2x speedup)
/usr/src/tensorrt/bin/trtexec \
  --onnx=best.onnx \
  --saveEngine=damaged_box_fp16.trt \
  --fp16 \
  --workspace=4096

# Expected inference speed on Jetson Orin:
# - FP16: ~30-60 FPS at 640x640
# - FP32: ~15-30 FPS at 640x640
```

**Python inference code for Jetson:**

```python
from ultralytics import YOLO
import cv2

# Load TensorRT model
model = YOLO('damaged_box_fp16.trt')

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    results = model(frame, conf=0.5)
    annotated = results[0].plot()
    cv2.imshow('Damage Detection', annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
```

## 📥 Step 9: Download Models

In [ ]:
# Download trained models
from google.colab import files

print("📥 Downloading model files...\n")
print("="*70)

# PyTorch model
print("1️⃣ Downloading PyTorch model (best.pt)...")
files.download(str(results_dir / "weights" / "best.pt"))

# ONNX model
print("2️⃣ Downloading ONNX model (best.onnx)...")
files.download(onnx_file)

print("\n" + "="*70)
print("✅ Files downloaded to your computer!")
print("="*70)

## 🎉 Summary & Next Steps

### ✅ What We Accomplished:

1. ✅ Downloaded complete damaged box dataset from Roboflow
2. ✅ Verified all images (torn, crushed, damaged)
3. ✅ Trained YOLOv8 model (50 epochs)
4. ✅ Achieved validation metrics (check above)
5. ✅ Exported to ONNX for Jetson Orin
6. ✅ Downloaded models for deployment

---

### 📊 Model Files You Have:

- `best.pt` - PyTorch model (for Python inference)
- `best.onnx` - ONNX model (for TensorRT conversion)

---

### 🚀 Deployment Workflow:

```
Google Colab           Jetson Orin
    │                      │
    ├─ Train YOLOv8       │
    ├─ Export ONNX  ──────┼─→ Convert to TensorRT
    └─ Download            └─→ Real-time inference (30-60 FPS)
```

---

### 💡 Tips to Improve Accuracy:

1. **Increase epochs**: Change `epochs=50` to `epochs=100`
2. **Try larger model**: Use `yolov8s.pt` or `yolov8m.pt`
3. **Data augmentation**: Roboflow has built-in augmentation
4. **Fine-tune**: Adjust learning rate and batch size

---

### 📚 Resources:

- **Dataset**: https://universe.roboflow.com/project-33xgh/damaged-box-detection
- **YOLOv8 Docs**: https://docs.ultralytics.com/
- **Jetson Setup**: https://developer.nvidia.com/embedded/jetson-orin

---

**🎊 Congratulations! Your damaged box detection model is ready for deployment!**

---